In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from scipy.stats import pearsonr

# === CONFIGURATION ===
root_dir = '/exports/eddie/scratch/s2556897/05_TSP_Deconvolution_v2'
results_dir = os.path.join(root_dir, 'Results-Pearson-PredictedVSActual')
os.makedirs(results_dir, exist_ok=True)

method_map = ['BayesPrism', 'nuSVR', 'ReDeconv', 'CIBERSORTx', 'MuSiC', 'NNLS', 'QP']

# Map proportions file → input directories
config_map = {
    "TSP-BDa_Outer_Random_v2C_New_All-Proportions.txt": [
        "TSP-BDa_Outer_300_1500_10-Random_v2",
        "TSP-BDa_Outer_1000_3000_10-Random_v2",
        "TSP-BDa_Outer_3000_5000_10-Random_v2"
    ],
    "TSP-BDa_Inner_Random_v2C_New_All-Proportions.txt": [
        "TSP-BDa_Inner_300_1500_10-Random_v2",
        "TSP-BDa_Inner_1000_3000_10-Random_v2",
        "TSP-BDa_Inner_3000_5000_10-Random_v2"
    ],
    "TSP-HBA_Inner_Random_v2C_New_All-Proportions.txt": [
        "TSP-HBA_Inner_300_1500_10-Random_v2",
        "TSP-HBA_Inner_1000_3000_10-Random_v2",
        "TSP-HBA_Inner_3000_5000_10-Random_v2"
    ]
}

all_results = []

# === MAIN LOOP ===
for prop_file, input_dirs in config_map.items():
    prop_path = os.path.join(root_dir, prop_file)

    print(f"\n=== Processing proportions file: {prop_file} ===")
    prop_df = pd.read_csv(prop_path, sep='\t', index_col=0)

    # Drop TotalCells if present
    if "TotalCells" in prop_df.columns:
        prop_df = prop_df.drop(columns=["TotalCells"])

    # Normalize
    prop_df = prop_df.div(prop_df.sum(axis=1), axis=0).multiply(100).round(5)
    
    for input_dir in input_dirs:
        decon_dir = os.path.join(root_dir, input_dir)
        decon_files = sorted(glob(os.path.join(decon_dir, '*_modified.txt')))

        if not decon_files:
            print(f" No decon files found in {decon_dir}")
            continue

        for file_path in decon_files:
            try:
                filename = os.path.basename(file_path)
                raw_method = filename.replace('_modified.txt', '')
                method = next((m for m in method_map if m in raw_method), raw_method)

                plot_name_base = f"{prop_file.replace('.txt','')}_{input_dir}_{method}"
                plot_png = os.path.join(results_dir, plot_name_base + ".png")
                plot_pdf = os.path.join(results_dir, plot_name_base + ".pdf")

                # Load decon results
                decon_df = pd.read_csv(file_path, sep='\t', index_col=0)

                # Align samples
                common_samples = prop_df.index.intersection(decon_df.index)
                if common_samples.empty:
                    print(f" No common samples in {file_path}")
                    continue

                # Align tissues (columns)
                common_cols = prop_df.columns.intersection(decon_df.columns)
                missing_in_decon = set(prop_df.columns) - set(decon_df.columns)
                missing_in_prop = set(decon_df.columns) - set(prop_df.columns)

                if missing_in_decon or missing_in_prop:
                    print(f" Column mismatch in {file_path}")
                    if missing_in_decon:
                        print(f"   Missing in decon: {missing_in_decon}")
                    if missing_in_prop:
                        print(f"   Missing in prop: {missing_in_prop}")

                if common_cols.empty:
                    print(f" No matching columns in {file_path}")
                    continue

                true_vals_df = prop_df.loc[common_samples, common_cols]
                pred_vals_df = decon_df.loc[common_samples, common_cols]

                # Flatten and remove zero-zero pairs
                true_vals_flat = true_vals_df.values.flatten()
                pred_vals_flat = pred_vals_df.values.flatten()
                mask_nonzero = ~((true_vals_flat == 0) & (pred_vals_flat == 0))
                true_vals_flat = true_vals_flat[mask_nonzero]
                pred_vals_flat = pred_vals_flat[mask_nonzero]

                # Pearson correlation
                r, p = pearsonr(true_vals_flat, pred_vals_flat)

                if p < np.finfo(float).tiny:
                    p_str = f"< {np.finfo(float).tiny:.1e}"
                else:
                    p_str = f"{p:.2e}"
                                
                print(f"{plot_name_base} — Pearson r = {r:.4f}, p = {p:.2e}")
                all_results.append([prop_file, input_dir, method, r, r**2, p])

                # Plot
                plt.figure(figsize=(8, 8))
                sns.set(style="white", context="talk")
                ax = sns.regplot(
                    x=true_vals_flat,
                    y=pred_vals_flat,
                    scatter_kws={'s': 30, 'alpha': 0.2, 'facecolor': 'black', 'edgecolors': 'none', 'rasterized': True},
                    line_kws={'color': 'crimson', 'lw': 2}
                )
                ax.set_xlim(0, 100)
                ax.set_ylim(0, 100)
                ax.set_xlabel('Actual Fraction (%)', fontsize=14, fontweight='bold')
                ax.set_ylabel('Predicted Fraction (%)', fontsize=14, fontweight='bold')
                ax.set_title(f'{prop_file}\n{input_dir} — {method}\nPearson r = {r:.2f}, p = {p:.2e}',
                             fontsize=14, fontweight='bold')
                ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
                sns.despine()
                plt.tight_layout()
                plt.savefig(plot_png, dpi=300, bbox_inches="tight")
                plt.savefig(plot_pdf, dpi=300, bbox_inches="tight")
                plt.close()

            except Exception as e:
                print(f" Error processing {file_path}: {e}")

# === SAVE SUMMARY ===
if all_results:
    # Convert results to DataFrame
    results_df = pd.DataFrame(all_results,
                              columns=["ProportionsFile", "InputDir", "Method", "Pearson_r", "R_squared", "p_value"])
    
    # Extract Matrix column from InputDir (keep BDa/HBA + Outer/Inner + size range)
    results_df["Matrix"] = results_df["InputDir"].apply(lambda x: "_".join(x.split("_")[0:4]))

    # Reorder columns
    results_df = results_df[["ProportionsFile", "Matrix", "Method", "Pearson_r", "R_squared", "p_value"]]
    
    results_csv = os.path.join(results_dir, "All_Pearson_Summary.csv")
    results_df.to_csv(results_csv, index=False, sep='\t')
    print(f"\n Saved Pearson summary table: {results_csv}")
else:
    print("\n⚠️ No results to save.")


=== Processing proportions file: TSP-BDa_Outer_Random_v2C_New_All-Proportions.txt ===
TSP-BDa_Outer_Random_v2C_New_All-Proportions_TSP-BDa_Outer_300_1500_10-Random_v2_CIBERSORTx — Pearson r = 0.7757, p = 0.00e+00
TSP-BDa_Outer_Random_v2C_New_All-Proportions_TSP-BDa_Outer_300_1500_10-Random_v2_NNLS — Pearson r = 0.4860, p = 0.00e+00
TSP-BDa_Outer_Random_v2C_New_All-Proportions_TSP-BDa_Outer_300_1500_10-Random_v2_QP — Pearson r = 0.4122, p = 0.00e+00
TSP-BDa_Outer_Random_v2C_New_All-Proportions_TSP-BDa_Outer_300_1500_10-Random_v2_ReDeconv — Pearson r = 0.7515, p = 0.00e+00
 Column mismatch in COO-Decon-Pseudobulks/TSP-BDa_Outer_300_1500_10-Random_v2/TSP-BDa_Outer_100each_seed42_filtered_Random_v1_prop_weights_MuSiC_modified.txt
   Missing in decon: {'leukocyte', 'kidney epithelial cell', 'myometrial cell', 'tissue-resident macrophage', 'luminal epithelial cell of mammary gland', 'enterocyte of epithelium proper of duodenum', 'progenitor cell of mammary luminal epithelium', 'taste recept

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import Normalize
import numpy as np
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# ------------------ Load CSV ------------------
file_path = 'Results-Pearson-PredictedVSActual/All_Pearson_Summary.csv'
df = pd.read_csv(file_path, sep='\t')

# ------------------ Pivot for heatmap ------------------
heatmap_data = pd.pivot_table(
    df,
    index='Matrix',
    columns='Method',
    values='Pearson_r',
    aggfunc='mean'
)

# ------------------ Extract Reference + MaxSig from Matrix ------------------
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))

annotations['Reference'] = idx_str.map(lambda x: "_".join(x.split('_')[:2]))

def get_maxsig(x):
    parts = x.split('_')
    if parts and parts[-1].isdigit():
        return int(parts[-1])
    return -1

annotations['MaxSig'] = idx_str.map(get_maxsig)

# ------------------ Sort rows by Reference then MaxSig ------------------
heatmap_data = (
    heatmap_data
    .assign(Reference=annotations['Reference'], MaxSig=annotations['MaxSig'])
    .sort_values(['Reference', 'MaxSig'])
    .drop(columns=['Reference', 'MaxSig'])
)

# Recompute annotations aligned with sorted index
annotations = pd.DataFrame(index=heatmap_data.index)
idx_str = pd.Index(annotations.index.astype(str))
annotations['Reference'] = idx_str.map(lambda x: "_".join(x.split('_')[:2]))
annotations['MaxSig'] = idx_str.map(get_maxsig)

# ------------------ Reorder columns (fixed method order) ------------------
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
heatmap_data = heatmap_data.reindex(columns=method_order)

# ------------------ Palettes & LUTs ------------------
matrix_groups = sorted(annotations['Reference'].unique())
matrix_palette = sns.color_palette("Set2", n_colors=len(matrix_groups))
matrix_lut = dict(zip(matrix_groups, matrix_palette))

maxsig_keys = sorted(annotations['MaxSig'].unique())
maxsig_palette = sns.color_palette("Set3", n_colors=len(maxsig_keys))
maxsig_lut = dict(zip(maxsig_keys, maxsig_palette))

# Row colors only
row_colors = pd.DataFrame({
    'Reference': [matrix_lut[g] for g in annotations['Reference']],
    'MaxSig': [maxsig_lut[k] for k in annotations['MaxSig']]
}, index=annotations.index)[['Reference', 'MaxSig']]

# ------------------ Fixed color normalization ------------------
norm = Normalize(vmin=0.3, vmax=0.9)

# ------------------ Plot clustermap ------------------
g = sns.clustermap(
    heatmap_data,
    row_colors=row_colors,
    col_cluster=False,
    row_cluster=False,
    cmap='coolwarm',
    figsize=(8, 5),
    annot=True, fmt=".2f",
    annot_kws={"size": 10},
    cbar_pos=None,
    linewidths=0,
    linecolor='white',
    dendrogram_ratio=(0.02, 0.02),
    norm=norm
)

# === Axis styling ===
g.ax_heatmap.set_yticks([])
g.ax_heatmap.set_yticklabels([])
g.ax_heatmap.set_ylabel("Matrix", labelpad=30, fontsize=10)
g.ax_heatmap.yaxis.set_label_position("left")

g.ax_heatmap.set_xlabel("Deconvolution Tool", fontsize=10)
g.ax_heatmap.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=3, width=0.5, labelsize=10
)
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=30, ha="right")

# === Updated Colorbar ===
cbar_ax = g.fig.add_axes([0.88, 0.23, 0.03, 0.5])
sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=norm)
sm.set_array([])
cbar = g.fig.colorbar(sm, cax=cbar_ax)
cbar.ax.set_ylabel("Pearson r", fontsize=14, rotation=270, labelpad=20)
cbar.ax.tick_params(labelsize=13, width=0.8, length=4)
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Legends ===
matrix_patches = [Patch(facecolor=matrix_lut[g], label=g) for g in matrix_groups]
maxsig_patches = [Patch(facecolor=maxsig_lut[k], label=str(k)) for k in maxsig_keys]

legend_ax1 = g.fig.add_axes([0.02, 0.65, 0.20, 0.20])
legend_ax1.axis('off')
legend_ax1.legend(handles=matrix_patches, title='Reference', loc='center', fontsize=10, title_fontsize=11)

legend_ax2 = g.fig.add_axes([0.02, 0.45, 0.20, 0.20])
legend_ax2.axis('off')
legend_ax2.legend(handles=maxsig_patches, title='Max Signatures', loc='center', fontsize=10, title_fontsize=11)

# Clean up row color axes
g.ax_row_colors.set_xticks([])
g.ax_row_colors.set_yticks([])
g.ax_row_colors.set_xticklabels([])
g.ax_row_colors.set_yticklabels([])

plt.subplots_adjust(top=0.95, bottom=0.05, left=0.25, right=0.85)
g.fig.savefig("Heatmap_Random_Pearson_WithReference_MaxSig.svg", bbox_inches="tight")
plt.close(g.fig)
